# MAML — Model-Agnostic Meta-Learning for Fast Adaptation of Deep Networks (2017)

_Papers · Meta-learning, Bayesian & Robustness_

**Learn an initialization that a single gradient step can fine-tune to any new task.**

---

This notebook is a practice scaffold for the **MAML — Model-Agnostic Meta-Learning for Fast Adaptation of Deep Networks (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Sanity-check one inner gradient step

MAML's inner loop is a single gradient step on a task's support set: $\theta' = \theta - \alpha\nabla_\theta\mathcal{L}$. We verify the mechanics on a tiny linear model ($y = wx + b$) with MSE loss and $\alpha=0.05$. From $\theta=(0,0)$ the gradient is $(-7,-4)$, giving $\theta'=(0.35,0.20)$ and a lower post-step loss — exactly the descent the outer loop relies on.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import copy

torch.manual_seed(0)
np.random.seed(0)

# Sanity-check the worked example: one inner step, linear model, MSE, alpha=0.05.
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
alpha_we = 0.05

xw = torch.tensor([1.0, 2.0])
yw = torch.tensor([1.0, 3.0])

loss_we = ((w*xw + b - yw)**2).mean()
gw, gb = torch.autograd.grad(loss_we, [w, b])

# One inner step on each parameter.
wp = w.item() - alpha_we * gw.item()
bp = b.item() - alpha_we * gb.item()

print("worked example: loss=%.4f grad=(%.1f, %.1f)  theta'=(%.2f, %.2f)" % (
      loss_we.item(), gw.item(), gb.item(), wp, bp))
print("  loss after 1 step =", round(((wp*xw + bp - yw)**2).mean().item(), 4))
# worked example: loss=5.0000 grad=(-7.0, -4.0)  theta'=(0.35, 0.20)
#   loss after 1 step = 2.3062

### Step 2 — Define the model and the sinusoid tasks

We use the paper's sinusoid setup: a tiny MLP, and a task distribution of sine waves with random amplitude and phase. The `forward` function runs the network at **arbitrary** weights $\theta'$ (not just the module's own), which is what lets the outer loop differentiate through the inner-adapted weights. Each task draws a *support* set (for adaptation) and a fresh *query* set (for the meta-objective).

In [ ]:
# The model: a tiny multi-layer perceptron, composed with torch.nn (paper, Sec 5.1).
def make_net():
    return nn.Sequential(nn.Linear(1, 40), nn.ReLU(),
                         nn.Linear(40, 40), nn.ReLU(),
                         nn.Linear(40, 1))

# Functional forward so we can run the net at ARBITRARY weights theta' (not just nn's own).
def forward(params, x):
    w1, b1, w2, b2, w3, b3 = params
    h = torch.relu(F.linear(x, w1, b1))
    h = torch.relu(F.linear(h, w2, b2))
    return F.linear(h, w3, b3)

# The sinusoid task distribution (paper Sec 5.1).
K, alpha, loss_fn = 10, 0.01, nn.MSELoss()

def sample_task():
    return np.random.uniform(0.1, 5.0), np.random.uniform(0, math.pi)

def task_points(A, phase, n):
    x = torch.tensor(np.random.uniform(-5.0, 5.0, (n, 1)), dtype=torch.float32)
    return x, A * torch.sin(x + phase)

### Step 3 — The bi-level meta-training loop

This is the heart of MAML. For each task we take an inner step on the support set with `create_graph=True`, which keeps that step in the autograd graph so the meta-gradient can flow **through** it (the second-order term). The adapted weights are computed as a plain expression `p - alpha*g` — *not* an optimizer step — and the outer loss is the query loss evaluated at those adapted weights. Backprop on the summed query loss then updates the shared initialization.

In [ ]:
# MAML meta-training: the NOVEL bi-level update, built by hand.
net = make_net()
meta_params = [p.clone().detach().requires_grad_(True) for p in net.parameters()]
meta_opt = torch.optim.Adam(meta_params, lr=1e-3)   # outer step size beta

for it in range(2000):
    meta_opt.zero_grad()
    meta_loss = 0.0
    for _ in range(4):                               # a batch of tasks
        A, phase = sample_task()
        xs, ys = task_points(A, phase, K)            # support set
        xq, yq = task_points(A, phase, K)            # query set (fresh!)
        # INNER LOOP: theta'_i = theta - alpha * grad,  create_graph=True keeps the
        # inner step in the graph so the META-gradient flows THROUGH it (second order).
        inner_loss = loss_fn(forward(meta_params, xs), ys)
        grads = torch.autograd.grad(inner_loss, meta_params, create_graph=True)
        adapted = [p - alpha*g for p, g in zip(meta_params, grads)]   # NOT an optimizer step
        # OUTER LOOP contribution: query loss AT the adapted weights theta'_i.
        meta_loss = meta_loss + loss_fn(forward(adapted, xq), yq)
    (meta_loss / 4).backward()                       # Eqn. 1: grad through theta'_i
    meta_opt.step()

### Step 4 — Train a joint-pretraining baseline

To show that MAML's benefit comes from its objective and not just from seeing the tasks, we train the *same* network jointly on all tasks with one head and no inner loop. This minimizes loss at $\theta$ itself, so it settles on the average sine curve — good on no single task.

In [ ]:
# Baseline: the SAME net pretrained JOINTLY on all tasks (no inner loop, one head).
torch.manual_seed(0)
np.random.seed(0)
joint = make_net()
joint_opt = torch.optim.Adam(joint.parameters(), lr=1e-3)

for it in range(2000):
    joint_opt.zero_grad()
    A, phase = sample_task()
    x, y = task_points(A, phase, 2*K)
    loss_fn(joint(x), y).backward()
    joint_opt.step()

### Step 5 — Evaluate one-step adaptation on held-out tasks

The real test: on 200 fresh tasks, take a **single** gradient step from each initialization and measure the query MSE. The MAML init adapts far better in one step than the jointly pretrained init — because MAML explicitly placed $\theta$ where one step reaches each task quickly, while the joint init sits at the average and barely moves.

In [ ]:
# Evaluate: adapt ONE gradient step from each init on held-out tasks.
np.random.seed(1234)
maml_post, joint_post = [], []

for _ in range(200):
    A, phase = sample_task()
    xs, ys = task_points(A, phase, K)
    xq, yq = task_points(A, phase, K)

    # MAML init: one inner step, then score on query.
    p = [q.clone().detach().requires_grad_(True) for q in meta_params]
    g = torch.autograd.grad(loss_fn(forward(p, xs), ys), p)
    p2 = [q - alpha*gg for q, gg in zip(p, g)]
    maml_post.append(loss_fn(forward(p2, xq), yq).item())

    # Joint init: one SGD step of the pretrained net.
    m = copy.deepcopy(joint)
    o = torch.optim.SGD(m.parameters(), lr=alpha)
    o.zero_grad()
    loss_fn(m(xs), ys).backward()
    o.step()
    joint_post.append(loss_fn(m(xq), yq).item())

print("\nHeld-out sinusoid tasks (200), mean query MSE after ONE gradient step:")
print("  MAML init   : %.4f" % np.mean(maml_post))
print("  Joint init  : %.4f" % np.mean(joint_post))
# MAML init   : ~1.34      <- adapts far better in one step
# Joint init  : ~2.88
# (Our small run, not the paper's reported numbers. Exact values vary by seed/hardware.)

## Visualize the data & results

_After meta-training, how fast does each initialization adapt to a NEW sinusoid task as we take more inner gradient steps?_

### Step 6 — Meta-train MAML and the baseline (self-contained)

The visualization re-runs both training procedures in a self-contained cell (its own imports and compact helpers) so it works on its own. It meta-trains MAML for 3000 iterations with the second-order inner step (`create_graph=True`), then trains the joint-pretraining baseline on the same task stream.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import copy

torch.manual_seed(0)
np.random.seed(0)

def make_net():
    return nn.Sequential(nn.Linear(1, 40), nn.ReLU(),
                         nn.Linear(40, 40), nn.ReLU(), nn.Linear(40, 1))

def forward(p, x):
    w1, b1, w2, b2, w3, b3 = p
    h = torch.relu(F.linear(x, w1, b1))
    h = torch.relu(F.linear(h, w2, b2))
    return F.linear(h, w3, b3)

def sample_task():
    return np.random.uniform(0.1, 5.0), np.random.uniform(0, math.pi)

def pts(A, ph, n):
    x = torch.tensor(np.random.uniform(-5, 5, (n, 1)), dtype=torch.float32)
    return x, A * torch.sin(x + ph)

K, alpha, lf = 10, 0.01, nn.MSELoss()

# meta-train MAML (second order, create_graph=True)
net = make_net()
mp = [p.clone().detach().requires_grad_(True) for p in net.parameters()]
opt = torch.optim.Adam(mp, lr=1e-3)
for it in range(3000):
    opt.zero_grad()
    ml = 0.0
    for _ in range(4):
        A, ph = sample_task()
        xs, ys = pts(A, ph, K)
        xq, yq = pts(A, ph, K)
        g = torch.autograd.grad(lf(forward(mp, xs), ys), mp, create_graph=True)
        ad = [p - alpha*gg for p, gg in zip(mp, g)]
        ml = ml + lf(forward(ad, xq), yq)
    (ml / 4).backward()
    opt.step()

# joint pretrain baseline
torch.manual_seed(0)
np.random.seed(0)
jn = make_net()
jo = torch.optim.Adam(jn.parameters(), lr=1e-3)
for it in range(3000):
    jo.zero_grad()
    A, ph = sample_task()
    x, y = pts(A, ph, 2*K)
    lf(jn(x), y).backward()
    jo.step()

### Step 7 — Plot the adaptation curves

Finally we average, over 100 held-out tasks, the query MSE as we take up to 5 inner gradient steps from each initialization. The MAML curve drops sharply after a single step and keeps improving; the joint-init curve starts similar but barely descends — visual confirmation that MAML's initialization is tuned for fast few-step adaptation.

In [ ]:
# average adaptation curve over 100 held-out tasks, 5 inner steps
np.random.seed(99)
STEPS, NT = 5, 100
maml_c = np.zeros(STEPS+1)
joint_c = np.zeros(STEPS+1)

for _ in range(NT):
    A, ph = sample_task()
    xs, ys = pts(A, ph, K)
    xq, yq = pts(A, ph, K)

    # MAML init: take STEPS inner gradient steps, recording query loss each time.
    p = [q.clone().detach().requires_grad_(True) for q in mp]
    with torch.no_grad():
        maml_c[0] += lf(forward(p, xq), yq).item()
    for s in range(STEPS):
        g = torch.autograd.grad(lf(forward(p, xs), ys), p)
        p = [(q - alpha*gg).detach().requires_grad_(True) for q, gg in zip(p, g)]
        with torch.no_grad():
            maml_c[s+1] += lf(forward(p, xq), yq).item()

    # Joint init: same number of SGD steps from the pretrained net.
    m = copy.deepcopy(jn)
    o = torch.optim.SGD(m.parameters(), lr=alpha)
    with torch.no_grad():
        joint_c[0] += lf(m(xq), yq).item()
    for s in range(STEPS):
        o.zero_grad()
        lf(m(xs), ys).backward()
        o.step()
        with torch.no_grad():
            joint_c[s+1] += lf(m(xq), yq).item()

maml_c /= NT
joint_c /= NT
print("MAML  init:", [round(v, 3) for v in maml_c])
print("Joint init:", [round(v, 3) for v in joint_c])
# MAML  init: [3.196, 0.944, 0.741, 0.68, 0.617, 0.597]
# Joint init: [3.374, 2.737, 2.529, 2.438, 2.371, 2.322]
# One inner step: MAML 0.944 vs joint 2.737. Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The first-order ablation. You have a working second-order MAML. Switch the inner gradient
            call from create_graph=True to create_graph=False (first-order MAML) and
            retrain, everything else identical. What term did you just drop, and what do you expect to happen to
            adaptation quality and speed?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Locate the inner step: grads = torch.autograd.grad(support_loss, params, create_graph=True). Set it to False. — _create_graph=False detaches the inner step from the graph, so the outer .backward() stops at the adapted weights and never differentiates through the inner gradient._
- Identify the dropped term: the factor $I - \alpha\nabla^2_\theta\mathcal{L}^{\text{sup}}$ collapses to $I$, so the Hessian-vector product (the second-order term) is gone. — _The meta-gradient becomes just $\nabla_{\theta'}\mathcal{L}^{\text{qry}}(\theta')$ &mdash; the gradient at the adapted weights, treated as if $\theta'$ did not depend on $\theta$._
- Retrain and compare: expect adaptation that is close to full MAML, at lower cost per meta-step. — _&sect;5.2 reports the first-order approximation performs "nearly the same" with "roughly" a "33% speed-up" &mdash; the backward pass through the inner gradient is skipped._

**Answer:** You dropped the second-order (Hessian-vector) term: with create_graph=False
                 the meta-gradient no longer flows through the inner step, collapsing $I-\alpha\nabla^2\mathcal{L}$
                 to $I$. This is exactly FOMAML. The paper (&sect;5.2) reports it is "nearly the same" in quality
                 with "roughly" a "33% speed-up." So the second-order term helps a little but is not essential
                 &mdash; the bulk of MAML's benefit comes from the bi-level objective itself, not from the
                 Hessian.

</details>

**Problem 2.** Why does a network jointly pretrained on all sine tasks (one head, no inner loop) adapt
            poorly in one gradient step, while MAML's init adapts well &mdash; even though both saw the
            same task distribution?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write each objective. Joint: $\min_\theta \mathbb{E}_i\,\mathcal{L}_{\mathcal{T}_i}(f_\theta)$ &mdash; loss at $\theta$ itself. MAML: $\min_\theta \mathbb{E}_i\,\mathcal{L}_{\mathcal{T}_i}(f_{\theta'_i})$ &mdash; loss after a step. — _Joint minimizes loss before any adaptation; MAML minimizes loss after one adaptation step. Different objectives, different optima._
- Reason about the joint optimum on sinusoids: amplitudes and phases vary, so the average-fitting solution is roughly the flat mean curve. One small step barely moves it toward any specific sine. — _Minimizing average loss lands "in the middle"; the middle is not positioned for fast per-task descent._
- Reason about the MAML optimum: $\theta$ is placed so that one step's gradient points sharply at whichever task appeared. The init is shaped for fast descent, not for low pre-step loss. — _MAML explicitly rewards post-step loss, so the gradient geometry around $\theta$ is tuned for one-step adaptation._

**Answer:** Because they optimize different objectives. Joint pretraining minimizes loss at
                 $\theta$, so it settles on the average curve &mdash; good on no task, and one step barely moves
                 it. MAML minimizes loss after one step, so it places $\theta$ where a single gradient
                 step reaches each task quickly. Same data, different target: "be good now" versus "be one step
                 from good." Our run below shows the gap: after one step the MAML init reaches far lower
                 held-out error than the jointly pretrained init.

</details>

**Problem 3.** In the worked example you had $\theta=(0,0)$, $\alpha=0.05$, points $x=[1,2]$, $y=[1,3]$, giving
            gradient $(-7,-4)$ and $\theta'_i=(0.35,0.20)$ with post-step loss $\approx 2.31$. Suppose you
            instead took a much larger inner step, $\alpha=0.40$. Compute $\theta'_i$ and the new loss. What
            does this say about choosing $\alpha$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply the inner update with $\alpha=0.40$: $w' = 0 - 0.40\cdot(-7) = 2.8$, $b' = 0 - 0.40\cdot(-4) = 1.6$. — _Same gradient $(-7,-4)$; only the step size changed, so $\theta'_i=(2.8,1.6)$._
- Compute the new loss: predictions $[2.8+1.6,\; 5.6+1.6] = [4.4, 7.2]$; residuals $[3.4, 4.2]$; loss $= \tfrac12(3.4^2 + 4.2^2) = \tfrac12(11.56 + 17.64) = 14.6$. — _The step overshot the minimum: loss jumped from $5.0$ to $14.6$, worse than before the step._
- Conclude: too large an $\alpha$ overshoots; the inner step must be small enough that one step improves the task loss. — _MAML's meta-update assumes the inner step is a sensible descent step; an overshooting $\alpha$ poisons the post-adaptation loss the outer loop is trying to lower._

**Answer:** With $\alpha=0.40$: $\theta'_i=(2.8,1.6)$ and the loss rises from $5.0$ to $14.6$
                 &mdash; the step overshot. The inner step size $\alpha$ must be small enough that one gradient
                 step actually lowers the task loss (the paper uses $\alpha=0.01$ for sinusoids). MAML's outer
                 loop minimizes the post-step loss, so an $\alpha$ that overshoots hands it a worse
                 objective and destabilizes meta-training. Contrast with the original $\alpha=0.05$, which
                 dropped the loss to $2.31$.

</details>